# Predictive Modeling: Borrower Default Risk
## Al-Rajab Lenders — Credit Risk Analysis

**Purpose:** Build a logistic regression model to predict whether a borrower will experience
serious delinquency (90+ days late) within 2 years, using the cleaned dataset.

**Key challenge:** The target variable is imbalanced — only ~6.68% of borrowers actually default.
A naive "always predict no default" model would show 93.3% accuracy while being completely
useless, so this notebook uses precision, recall, and a confusion matrix rather than relying on
accuracy alone.


## 1. Load Data and Prepare Features

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

df = pd.read_csv('credit_risk_cleaned.csv')

# Separate features (X) from target (y)
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True) * 100}")


Features shape: (149999, 11)
Target distribution:
SeriousDlqin2yrs
0    93.315955
1     6.684045
Name: proportion, dtype: float64


## 2. Train-Test Split

We hold out 20% of the data as a test set the model never sees during training. `stratify=y`
ensures both the training and test sets preserve the same ~6.68% default rate — otherwise a
random split could accidentally leave too few (or too many) default cases in one side.


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"\nTraining set default rate: {round(y_train.mean()*100, 2)}%")
print(f"Test set default rate: {round(y_test.mean()*100, 2)}%")


Training set: 119999 rows
Test set: 30000 rows

Training set default rate: 6.68%
Test set default rate: 6.68%


## 3. Baseline: The "Dumb Model" (Always Predict No Default)

Before training a real model, we calculate what accuracy a naive "always predict no default"
model would achieve — this is the benchmark any real model must clearly beat to prove it's
actually useful, not just exploiting the class imbalance.


In [3]:
# A model that always predicts 0 (no default)
dummy_predictions = [0] * len(y_test)

dummy_accuracy = accuracy_score(y_test, dummy_predictions)
dummy_recall = recall_score(y_test, dummy_predictions, zero_division=0)

print(f"Dummy model accuracy: {round(dummy_accuracy*100, 2)}%")
print(f"Dummy model recall (catches actual defaulters): {round(dummy_recall*100, 2)}%")
print("\nThe dummy model looks 'accurate' but catches ZERO actual defaulters -- useless in practice.")


Dummy model accuracy: 93.32%
Dummy model recall (catches actual defaulters): 0.0%

The dummy model looks 'accurate' but catches ZERO actual defaulters -- useless in practice.


## 4. Feature Scaling

Our features have very different scales -- `MonthlyIncome` is in the thousands, while
`RevolvingUtilizationOfUnsecuredLines` is a small decimal. Without scaling, logistic regression's
internal optimization can fail to converge properly, since large-scale features can dominate the
math. `StandardScaler` transforms every feature to have mean 0 and standard deviation 1, putting
them all on equal footing. The scaler is fit only on the training data, then applied to both sets
-- this avoids leaking any information from the test set into training.


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")


Features scaled successfully.


## 5. Train Logistic Regression Model

Logistic Regression is a standard, interpretable starting model for binary classification
problems like this one (default vs. no default). `class_weight='balanced'` tells the model to pay
more attention to the minority class (defaulters) during training, rather than being dominated by
the majority "no default" class.


In [5]:
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Model trained successfully -- no convergence warning this time.")


Model trained successfully -- no convergence warning this time.


## 5. Evaluate the Model

We compare accuracy, precision, and recall against the dummy baseline above. Precision answers
"of the borrowers we flagged as high-risk, how many actually defaulted?" Recall answers "of the
borrowers who actually defaulted, how many did we correctly catch?" For a lending business, recall
is often prioritized -- missing an actual defaulter is usually more costly than a false alarm.


In [6]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Model Accuracy:  {round(accuracy*100, 2)}%  (dummy model: {round(dummy_accuracy*100, 2)}%)")
print(f"Model Precision: {round(precision*100, 2)}%")
print(f"Model Recall:    {round(recall*100, 2)}%  (dummy model: {round(dummy_recall*100, 2)}%)")
print(f"Model F1 Score:  {round(f1*100, 2)}%")

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(f"True Negatives (correctly predicted no default):  {cm[0][0]}")
print(f"False Positives (predicted default, but didn't):  {cm[0][1]}")
print(f"False Negatives (predicted no default, but did):  {cm[1][0]}")
print(f"True Positives (correctly predicted default):     {cm[1][1]}")


Model Accuracy:  76.96%  (dummy model: 93.32%)
Model Precision: 17.69%
Model Recall:    66.98%  (dummy model: 0.0%)
Model F1 Score:  27.98%

--- Confusion Matrix ---
True Negatives (correctly predicted no default):  21745
False Positives (predicted default, but didn't):  6250
False Negatives (predicted no default, but did):  662
True Positives (correctly predicted default):     1343


## 6. Feature Importance

Which factors does the model rely on most heavily? Logistic regression coefficients show
direction and relative strength -- a positive coefficient means higher values of that feature
push the prediction toward "default," and larger magnitude means stronger influence.


In [7]:
coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(coefficients)


                                 feature  coefficient
2   NumberOfTime30-59DaysPastDueNotWorse     2.187760
6                NumberOfTimes90DaysLate     2.118221
8   NumberOfTime60-89DaysPastDueNotWorse     0.174121
7           NumberRealEstateLoansOrLines     0.082415
9                     NumberOfDependents     0.068288
0   RevolvingUtilizationOfUnsecuredLines     0.000559
5        NumberOfOpenCreditLinesAndLoans    -0.016177
10                    Income_Was_Missing    -0.023184
3                              DebtRatio    -0.076310
4                          MonthlyIncome    -0.239647
1                                    age    -0.434907


## Summary

| Metric | Dummy Model | Logistic Regression |
|---|---|---|
| Accuracy | ~93.3% | (see output above) |
| Recall (catches actual defaulters) | 0% | (see output above) |

The dummy model's high accuracy is meaningless -- it catches zero actual defaulters. The real
value of the logistic regression model is its recall: its ability to actually identify borrowers
who go on to default, which is the entire business purpose of this analysis.

**Next step:** Translate these results into business insights and recommendations for Al-Rajab
Lenders' loan approval process.
